# Where CMA-ES and BO fail on a needle landscape, and what fixes it

**Research question.** At a fixed budget of N = 300 evaluations, how do population size, restarts and acquisition choices change the probability that CMA-ES and GP-BO find a narrow or distant global basin, and why?

**Convention.** This notebook only loads saved results (`results/summary*/`, `results/raw*/`) and draws them. Runs, aggregation and figure code live in `run_experiments.py` and `code/` (`analysis.py`, `report.py`). Findings live here, in markdown: prediction, result with CI, explanation, caveat.

- `FAST = True`: smoke test, N = 100, 5 seeds, from `results/*_fast/`. Every figure drawn from it carries a "SMOKE TEST" stamp.
- `FAST = False`: full results (N = 300, 50 seeds) from `results/summary/`.

The status table says which experiments have results and gives the command to produce the rest.

In [ ]:
FAST = True   # True: smoke test (N=100, 5 seeds) from results/*_fast.  False: full results from results/summary.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd().resolve()   # find the repo whatever folder the kernel starts in (notebooks/, repo root, or its parent)
ROOT = next(p for p in [here, *here.parents, *(d for d in here.iterdir() if d.is_dir())] if (p / "code" / "report.py").exists())
sys.path.insert(0, str(ROOT / "code"))
import report
M = report.Mode(FAST)
print(f"python = {sys.executable}  (should end in DSA4212/.venv/bin/python)")
print(f"mode = {'FAST (smoke test)' if FAST else 'FULL'}; N = {M.N}; seeds per cell = {M.n_seeds}; summaries: {M.summary_dir.relative_to(ROOT)}")

## 0. Status: which experiments have results

In [ ]:
report.status_table(M)

*Next: the headline findings, left open until the full runs exist.*

## 1. Summary

> **The FAST numbers in this notebook (N = 100, 5 seeds) are a smoke test of the pipeline, not results.** No finding below is written until the full run exists.

- **Baselines (Exp 0, full run, N = 300, 50 seeds):** default CMA-ES succeeded in 0/50 runs, default BO in 5/50 and random sampling in 2/50. The three Wilson intervals overlap pairwise, so the methods are not distinguishable from each other or from random sampling at this seed count. CMA-ES ended in the decoy in 48/50 runs; BO's 45 failures never sampled inside the well. See sections 3 and 4 (the explanations there are hypotheses).
- **Population size (Exp A):** to fill in after the full run.
- **Restarts (Exp B):** to fill in after the full run.
- **BO exploration (Exp C):** to fill in after the full run.
- **Unimodal control (Exp D):** to fill in after Exps A-C choose the settings and Exp D runs.
- **Basin width (Exp E):** to fill in after the full run.
- **Validation (Exp V):** to fill in after Exps A-C choose the settings and Exp V runs.
- **Use X when Y:** to fill in after the full run (section 7).

*Next: the landscape and the definitions every later number rests on.*

## 2. Setup: the landscape, success, outcomes, seeds

**Primary landscape.** A broad decoy bowl centred at $c$ plus a narrow, deep Gaussian well at $x_g$:

$$f(x) = \frac{\lVert x - c\rVert^2}{s} \;-\; A\,\exp\!\left(-\frac{\lVert x - x_g\rVert^2}{2w^2}\right)$$

with $c = 0$, $s = 10$, $A = 4$, $w = 0.3$ and the far well at $x_g = (3, 3)$, on the domain $[-5, 5]^2$. Exp E varies $w$ and moves the well to $(2, 0)$ ("near").

**Success.** A run succeeds when its best value has regret $\le \varepsilon$, where regret is $f_{\text{best}} - f^*$. Because regret needs the known minimum $f^*$, it is available only on benchmark functions.

- **Primary:** $\varepsilon = 0.05$, chosen from random sampling only: the largest value on a fixed grid at which uniform random sampling at N = 300 succeeds in at most 5% of 200 pilot seeds (9000-9199). It gave 5/200 = 2.5%. The success region is a disc of radius about 0.05 around $x^*$ (`docs/figures/epsilon_check.png`).
- **Validation functions** use the same procedure with a cap below the nearest non-global minimum's regret, so success implies the global basin: Rastrigin $\varepsilon = 0.2$ (cap 0.995), Schwefel $\varepsilon = 5$ (cap 118.4; its random success sits exactly on the 5% boundary, 10/200).

**Outcome labels** (every run gets exactly one):

- **success**: regret $\le \varepsilon$.
- **stuck in decoy**: the best point found does not lead to the well (local descent from it ends in the decoy).
- **unfinished**: the best point is in the well's basin but not within $\varepsilon$ of the minimum, so the run found the basin and did not finish refining.

**Fixed settings (frozen before the big runs).** Budget N = 300 for every CMA-ES vs BO comparison, restarts and BO initial design included. CMA-ES: $\sigma_0 = 2$, start $x_0 \sim U[-4, 4]^2$, restart triggers tolx = 3e-3 and tolfun = 1e-5, restarts redraw $x_0$ in the full box. BO: Matern-5/2 ARD GP fitted by maximum likelihood with noise fixed at 1e-6, lengthscale bounds [0.05, 20] domain units, 10 initial space-filling points, refit every 5 iterations.

**Seeds, stated once.** Main experiments use seeds 1000-1049 (FAST uses the first five, 1000-1004). Fresh seeds 2000-2049 are used only for the Exp E best-variant line. Pilot seeds 9000-9199 are used only to choose $\varepsilon$ and the restart triggers. A seed fixes the CMA-ES start point and its random stream, and the BO initial design and Thompson draws, so every variant on seed $k$ starts from the same point or the same initial design. A run is reproducible from (method, config, landscape, seed).

**Visual language.** Same contour map throughout: **star** = true minimum, **cross** = decoy centre. CMA-ES is blue, BO is orange, random sampling is grey. Success rates are bars with Wilson 95% intervals and k/n above each bar.

In [ ]:
report.landscape_figure(M)

*Next: what happens when the two default optimisers meet this landscape.*

## 3. Failure: the Exp 0 baselines

First three **illustrations** of single runs (they show what a run looks like and prove nothing about rates), then the aggregate over seeds.

**CMA-ES, seed 1000.** Samples coloured by generation (light = early), the diamond is the start, red ellipses are the 1-sigma search distribution at a few generations, and the right panel is sigma per generation. The path is re-run from the same seed and checked against the saved CSV (a warning appears on the figure if they differ).

In [ ]:
report.cmaes_illustration(M)

**BO, seed 1000.** The 10 initial design points are white squares and later points are coloured by the order they were chosen.

In [ ]:
report.bo_illustration(M)

**Four seeds each (1000-1003).** The first four seeds of the fixed list, not picked for how they look. Top row CMA-ES, bottom row BO, colour = evaluation order.

In [ ]:
report.seed_grid(M)

**Aggregate over seeds.** Success rate with Wilson 95% intervals (left) and what happened to each run (right). Difference rows use a Newcombe interval, which is unpaired and so conservative when seeds are matched.

In [ ]:
report.show_predictions(["H1"])
report.exp0_figure(M)

*The numbers below are from the saved full run (N = 300, 50 seeds, seeds 1000-1049; set `FAST = False` to see its figures). The FAST figures above are a smoke test.*

**Prediction H1** (shown above): default CMA-ES gets stuck in a decoy basin.

**Result (with CI and effect size).** On `primary_far_w0.3`, $\varepsilon = 0.05$, N = 300:

| method | success | Wilson 95% | stuck in decoy | unfinished | median regret |
| --- | --- | --- | --- | --- | --- |
| CMA-ES default | 0/50 | [0.000, 0.071] | 48 | 2 | 2.208 |
| BO default (EI, $\xi$ = 0) | 5/50 | [0.043, 0.214] | 45 | 0 | 2.208 |
| random sampling | 2/50 | [0.011, 0.135] | 11 | 37 | 1.114 |

Differences in success rate (Newcombe 95%, unpaired so conservative): BO minus CMA-ES +0.10 [+0.01, +0.21]; BO minus random +0.06 [-0.05, +0.18]; CMA-ES minus random -0.04 [-0.13, +0.04].

By the rule "claim a difference only when the intervals separate", **the three methods are not distinguishable from one another at 50 seeds, random sampling included**: all three Wilson intervals overlap pairwise. The BO minus CMA-ES interval only just excludes zero and is a weaker statement; it does not license ranking them. What the data do show is that none of the three usually succeeds: even BO's upper limit is 21%, about one run in five.

**H1.** On each of H1's "supported by" criteria the measurements fall on the supported side: low success (0/50); best point outside the global basin in 48 of 50 CMA-ES runs; sigma collapse (section 4). Nothing on the "contradicted by" side was seen (success is not high and sigma does not stay large). The pre-registration does not define "early" collapse, so section 4 reports the numbers and the reading is left to you. The status in `docs/scoreboard.csv` is yours to set.

**Explanation** (hypotheses, not tested here):

- *Hypothesis:* the decoy bowl slopes toward its centre from everywhere, so a CMA-ES run that does not sample the well in its first generations has no signal pointing at it; the search distribution then contracts on the decoy. The two "unfinished" runs fit this picture: an early sample (generation 0 and generation 6) landed on the well's flank, yet sigma still collapsed onto the decoy.
- *Hypothesis:* BO and random sampling succeed mainly when a space-filling sample happens to land in the well. Of BO's five successes, three first entered the well within the 10-point initial design (evaluations 5, 5 and 9), one at evaluation 20 and one at evaluation 115. Measured, not tested: in the 45 failures no sample ever entered the well.

**Caveat.** One landscape (well width 0.3, far well) and one $\varepsilon$; 50 seeds leaves wide intervals, so small differences are unresolved. The settings are frozen ($\sigma_0 = 2$, start $U[-4,4]^2$, BO with 10 initial points, EI with $\xi = 0$, lengthscale bound 20) and results may depend on them. BO reproduces in distribution, not bit for bit, so the exact count 5/50 could shift on a re-run. "Unfinished" means the best point lies in the well's basin, not that the run was converging there (random sampling's 37 are best points on the well's flank).

*Next: the diagnostics that say why the baselines behave this way.*

## 4. Why: diagnostics

Two measured quantities, one per method. The mechanisms behind them are **hypotheses** until tested.

**CMA-ES step size.** Sigma per generation for every seed. *Hypothesis (from `docs/direction.md` section 2, not tested here):* inside a basin consecutive steps are anticorrelated, cumulative step-size adaptation shrinks sigma, and the search contracts and cannot leave. Measured: sigma, and where the best point ends up (section 3 outcomes). H1 is supported only by low success together with early sigma collapse and a best point outside the global basin.

In [ ]:
report.sigma_figure(M)

**BO fitted lengthscale versus the well width $w$.** Left: the smallest fitted lengthscale along a run on the primary landscape, against $w$ and the fitting upper bound. Right: its final value against $w$ across the far-well widths. *Hypothesis (not tested here):* when the lengthscale is much longer than $w$ the GP smooths over the well, gives it low uncertainty, and EI never samples it. Measured: the lengthscale and whether samples stay spread out; a causal claim would need a test that changes the lengthscale and nothing else. A lengthscale pinned at the upper bound also reflects the bound chosen before the runs.

In [ ]:
report.lengthscale_figure(M)

*The numbers below are from the saved full run (N = 300, 50 seeds); set `FAST = False` to see its figures.*

**Result (measured).**

- *CMA-ES step size.* Sigma starts at 2, rises above it in the first generations (median peak 2.24), then decays steadily in every run. It first falls below 0.2 (a tenth of the start value) at generation 19 (median; range 14-31) of 50, and below 0.02 at generation 35 (range 29-49). Median final sigma is 0.0025. From evaluation 100 on, at least 97.5% of every run's samples lie within 0.5 of the decoy centre, and the closest any CMA-ES sample comes to the well centre is 1.4 (median over runs).
- *BO lengthscale.* In 42 of 50 runs the smallest fitted lengthscale reaches the upper bound (20, twice the domain width), first at evaluation 15 (median; range 15-130), and it ends there. None of those 42 succeeded. Five of the other eight runs succeeded: their final lengthscale is 0.05-0.98, i.e. 0.17 to 3.3 times the well width $w = 0.3$.
- *BO sampling.* In the 45 failures no sample ever entered the well (closest approach 1.41, median), and about 99% of evaluations from 100 on are within 0.5 of the decoy centre. All 5 runs that did put a sample inside the well reached $\varepsilon$.

**Explanation** (hypotheses, not tested here):

- *CMA-ES hypothesis (from `docs/direction.md` section 2):* cumulative step-size adaptation shrinks sigma when consecutive steps are anticorrelated, as they are inside a basin, so the search contracts and cannot leave. The steady geometric decay in all 50 runs is consistent with this; no run held sigma fixed, so the mechanism itself is untested. Whether collapse is "early" is a reading of the numbers above (in the median run sigma is below 0.2 after 38% of the generations and below 0.02 after 70%), not a pre-registered threshold.
- *BO hypothesis A:* a lengthscale much longer than $w$ makes the surrogate smooth over the well, so EI never samples it.
- *BO hypothesis B (the reverse direction):* a run that has not yet sampled the well only ever sees the smooth bowl, so maximum likelihood picks a long lengthscale; the long lengthscale is then a symptom of missing the well, not its cause. The data cannot separate A from B: runs with a long lengthscale are exactly the runs without a well sample.
- *What would test them:* changing the lengthscale (for example capping it) while holding everything else fixed, or raising exploration ($\xi$, $\beta$) and checking whether samples reach the well; the latter is Exp C.

**Caveat.** Where the lengthscale sits at the bound, the likelihood wanted a longer value than the frozen bound allows, so "at the bound" is partly a property of the bound. The BO split (all 5 runs with a well sample succeeded, none of the 45 others sampled it) says discovery, not refinement, is where BO fails here, but with 5 successes it is a description of this baseline, not an estimate of a rate. BO is reproducible in distribution only.

*Next: one factor at a time, to see which design choices change these failures.*

## 5. Experiment cards

One factor changes per experiment; every variant uses the same N and the same seeds. Each card has the same five parts: the pre-registered **prediction** (read from `docs/predictions.md`, unedited), a **figure** (success rate with Wilson 95% interval, same style as section 3, plus a table with the difference against the baseline), the **result** with CI and effect size, the **explanation**, and the **caveat**. A card's result and explanation are filled in only when its full results exist.

### Exp A: population size $\lambda$ (1x, 2x, 4x default: 6, 12, 24)

Produce results: `python run_experiments.py --full --exp expA && python run_experiments.py --aggregate`

Also report generations completed per $\lambda$ (the table shows the median).

In [ ]:
report.show_predictions(['H2', 'M1'])
report.experiment_card("expA", M)

**Result (with CI and effect size):** *To fill in after the full run.*

**Explanation** (untested explanations are labelled *hypothesis*): *To fill in after the full run.*

**Caveat:** *To fill in after the full run.*

### Exp B: restarts (none, IPOP, BIPOP; total budget fixed at N)

Produce results: `python run_experiments.py --full --exp expB && python run_experiments.py --aggregate`

Restart triggers are frozen at tolx = 3e-3, tolfun = 1e-5. **Must report** restart firing rate per $\lambda$ (pilot, no-restart runs: 100% at $\lambda$ = 6, 84% at 12, 0% at 24). At $\lambda$ = 24 restarts never fire, so IPOP/BIPOP started there equal no restarts.

In [ ]:
report.show_predictions(['H3', 'M2'])
report.experiment_card("expB", M)

**Result (with CI and effect size):** *To fill in after the full run.*

**Explanation** (untested explanations are labelled *hypothesis*): *To fill in after the full run.*

**Caveat:** *To fill in after the full run.*

### Exp C: BO exploration (EI with $\xi$, LCB with $\beta$, Thompson sampling, MAP-fit arm)

Produce results: `python run_experiments.py --full --exp expC && python run_experiments.py --aggregate`

Also report sample spread, the fitted lengthscale against $w$ (section 4) and the Cholesky jitter-fallback count.

**Link to Exp 0.** B1 (post hoc, from Exp 0) says default BO fails at discovery; H4 tests whether more exploration fixes that; M4 says any gain comes from discovery and not from a shorter lengthscale. M5 uses the `bo_map` arm to separate "long lengthscale causes the failure" from "it is a symptom". Same shape as CMA-ES: H1, M1, H2. Prediction text is in `docs/predictions_addendum.md`.

In [ ]:
report.show_predictions(['H4', 'B1', 'M4', 'M5'])
report.experiment_card("expC", M)

**Result (with CI and effect size):** *To fill in after the full run.*

**Explanation** (untested explanations are labelled *hypothesis*): *To fill in after the full run.*

**Caveat:** *To fill in after the full run.*

### Exp D: unimodal control (sphere)

Produce results: `to be added to `code/configs.py` once Exps A-C choose the best and worst settings`

Figure when configured: median regret curves with quartiles, plus final regret at equal N per setting.

In [ ]:
report.show_predictions(['H5'])
report.experiment_card("expD", M)

**Result (with CI and effect size):** *To fill in after the full run.*

**Explanation** (untested explanations are labelled *hypothesis*): *To fill in after the full run.*

**Caveat:** *To fill in after the full run.*

### Exp E: basin width

Produce results: `python run_experiments.py --full --exp expE_baseline && python run_experiments.py --aggregate`

The far well runs at $w \in \{0.1, 0.3, 0.6, 1.0\}$ and the near well only at $w \le 0.3$ (the near well's basin merges with the decoy at $w \ge 0.6$). One fixed $\varepsilon$ is used throughout. The best-variant line will be selected by the rule in `docs/pilot_log.md` (written before Exps A-C) and evaluated on fresh seeds 2000-2049; the baseline-only curves use seeds 1000-1049. **No prediction is pre-registered for Exp E.**

In [ ]:
report.experiment_card("expE_baseline", M)

**Result (with CI and effect size):** *To fill in after the full run.*

**Explanation** (untested explanations are labelled *hypothesis*): *To fill in after the full run.*

**Caveat:** *To fill in after the full run.*

### Exp V: validation on Rastrigin (funnel) and Schwefel (deceptive)

Produce results: `to be added to `code/configs.py` once Exps A-C choose the best and worst settings`

Figures when configured: success rate per function (separate panels) and the $\lambda$ gain (success-rate difference with a Newcombe interval, unpaired so conservative).

In [ ]:
report.show_predictions(['H6', 'M3'])
report.experiment_card("expV", M)

**Result (with CI and effect size):** *To fill in after the full run.*

**Explanation** (untested explanations are labelled *hypothesis*): *To fill in after the full run.*

**Caveat:** *To fill in after the full run.*

*Next: every pre-registered prediction in one table, with its status.*

## 6. Prediction scoreboard

Every prediction in `docs/predictions.md`, including any later amendments. **Statuses live in `docs/scoreboard.csv` and are changed by hand, never by code**; the default is "not yet run". A prediction that is contradicted, with the diagnostics that explain it, shows more understanding than a clean confirmation.

In [ ]:
report.scoreboard()

*Next: what the findings recommend, and how far they can be trusted.*

## 7. Verdict and limitations

**Use X when Y:** *to fill in after the full run.*

**Limitations.**

- **Low dimension:** the main setting is d = 2; the d = 5 check, if done, is one case.
- **Finite seeds:** 50 seeds gives a Wilson half-width of about 14% at a success rate near 0.5, so some differences will not be resolvable. A claim needs separated intervals; otherwise write "not distinguishable at this seed count".
- **One primary custom landscape**, plus two standard validation functions. A finding on the needle landscape need not transfer.
- **Regret needs a known minimum**, so success is defined only on benchmark functions, and $\varepsilon$ is a modelling choice (chosen from random sampling only).
- **Untested mechanisms are hypotheses.** Sigma collapse, lengthscale smoothing and selection averaging are explanations to be tested, not established facts.
- **Reproducibility:** A seed fixes the initial design and random streams. CMA-ES and random sampling reproduce exactly; BO reproduces in distribution, because floating-point differences compound through the model-fitting loop. Paired comparisons are paired on starting conditions only.
- **Frozen settings:** BO lengthscale bounds, CMA-ES start and restart triggers were fixed before the big runs; other choices could change the picture.